In [ ]:
%matplotlib inline

# NLP From Scratch: Translation with a Sequence to Sequence Network and Attention

Borrow from: [Sean Robertson](https://github.com/spro)

In this project we will be teaching a neural network to translate from French to English.

This is made possible by the simple but powerful idea of the [sequence to sequence network](https://arxiv.org/abs/1409.3215), in which two recurrent neural networks work together to transform one sequence to another. An encoder network condenses an input sequence into a vector, and a decoder network unfolds that vector into a new sequence.

To improve upon this model we'll use an [attention mechanism](https://arxiv.org/abs/1409.0473), which lets the decoder learn to focus over a specific range of the input sequence.

In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Loading data files

The data for this project is a set of many thousands of English to French translation pairs from [Tatoeba](https://tatoeba.org/). The file is a tab-separated list of translation pairs.

In [ ]:
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
# Turn a Unicode string to plain ASCII
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [ ]:
import os

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")
    # 尝试多个路径，兼容不同启动方式
    filepath = 'data/%s-%s.txt' % (lang1, lang2)
    if not os.path.exists(filepath):
        filepath = 'lab3/seq2seq/data/%s-%s.txt' % (lang1, lang2)
    lines = open(filepath, encoding='utf-8').\
        read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [ ]:
MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print("Sample pair:", random.choice(pairs))

## The Seq2Seq Model

A Sequence to Sequence network (Encoder-Decoder) consists of two RNNs. The encoder reads an input sequence and outputs a context vector; the decoder reads that vector to produce an output sequence.

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # 尝试多个路径，兼容不同启动方式
    filepath = 'data/%s-%s.txt' % (lang1, lang2)
    if not os.path.exists(filepath):
        filepath = 'lab3/seq2seq/data/%s-%s.txt' % (lang1, lang2)

    lines = open(filepath, encoding='utf-8').\
        read().strip().split('\n')

    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

### Simple RNN Decoder (without Attention)

The decoder uses the encoder's last hidden state as its initial hidden state. At each step it takes the previous token and hidden state, and produces the next token.

**TODO: Complete the `forward_step` method.**

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None

    def forward_step(self, input, hidden):
        embedded = self.embedding(input)
        output, hidden = self.gru(embedded, hidden)
        output = self.out(output)
        return output, hidden

### Bahdanau Attention (Additive Attention)

Attention allows the decoder to focus on different parts of the encoder output at each decoding step.

**TODO: Complete the `forward` method of `BahdanauAttention`.**

The attention mechanism computes:
1. **Energy scores**: $e_{ij} = V_a^T \tanh(W_a s_{i-1} + U_a h_j)$
   - $s_{i-1}$: decoder hidden state (query)
   - $h_j$: encoder output at position j (keys)
2. **Attention weights**: $\alpha_{ij} = \text{softmax}(e_{ij})$
3. **Context vector**: $c_i = \sum_j \alpha_{ij} h_j$

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)  # transform keys
        self.Ua = nn.Linear(hidden_size, hidden_size)  # transform query
        self.Va = nn.Linear(hidden_size, 1)            # compute score

    def forward(self, query, keys):
        # query: [batch_size, 1, hidden_size] — decoder hidden state
        # keys:  [batch_size, seq_len, hidden_size] — encoder outputs

        # Compute energy scores: Va * tanh(Wa(keys) + Ua(query))
        # Ua(query) broadcasts from [batch, 1, hidden] to [batch, seq_len, hidden]
        scores = self.Va(torch.tanh(self.Wa(keys) + self.Ua(query)))
        # scores: [batch_size, seq_len, 1]

        # Softmax over sequence length to get attention weights
        attn_weights = F.softmax(scores, dim=1)
        # attn_weights: [batch_size, seq_len, 1]

        # Weighted sum of encoder outputs to get context vector
        context = torch.bmm(attn_weights.transpose(1, 2), keys)
        # context: [batch_size, 1, hidden_size]

        return context, attn_weights


class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        # GRU input: [embedded; context] => 2 * hidden_size
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))

        # Compute attention context
        query = hidden.permute(1, 0, 2)  # [batch, 1, hidden]
        context, attn_weights = self.attention(query, encoder_outputs)

        # Concatenate embedded input and context vector
        input_gru = torch.cat((embedded, context), dim=2)  # [batch, 1, 2*hidden]

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)  # [batch, 1, output_size]

        return output, hidden, attn_weights

## Training

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)
    plt.xlabel('Epoch (per 100)')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.show()

In [ ]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
          print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
                           encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)
    return plot_losses

## Evaluation

In [ ]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn


def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

## Experiment 1: Simple Seq2Seq (without Attention)

In [ ]:
hidden_size = 64
batch_size = 32
n_epochs_rnn = 200

# Reload data for consistent state
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder_rnn = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_rnn = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("=" * 50)
print("Training Simple Seq2Seq Model (RNN Decoder, no Attention)")
print("=" * 50)
print(f"Encoder: {encoder_rnn}")
print(f"Decoder: {decoder_rnn}")

losses_rnn = train(train_dataloader, encoder_rnn, decoder_rnn, n_epochs_rnn,
                   print_every=50, plot_every=50)

In [ ]:
encoder_rnn.eval()
decoder_rnn.eval()
print("\n" + "=" * 50)
print("Evaluation: Simple Seq2Seq (without Attention)")
print("=" * 50)
evaluateRandomly(encoder_rnn, decoder_rnn)

## Experiment 2: Seq2Seq with Attention

In [ ]:
# Reload data for consistent state
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder_attn = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_attn = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

print("=" * 50)
print("Training Seq2Seq Model WITH Attention")
print("=" * 50)
print(f"Encoder: {encoder_attn}")
print(f"Attention Decoder: {decoder_attn}")

n_epochs_attn = 200
losses_attn = train(train_dataloader, encoder_attn, decoder_attn, n_epochs_attn,
                    print_every=50, plot_every=50)

In [ ]:
encoder_attn.eval()
decoder_attn.eval()
print("\n" + "=" * 50)
print("Evaluation: Seq2Seq WITH Attention")
print("=" * 50)
evaluateRandomly(encoder_attn, decoder_attn)

## Comparison: Simple RNN vs Attention

In [ ]:
# Plot loss comparison
plt.figure(figsize=(10, 5))
plt.plot(losses_rnn, label='Seq2Seq w/o Attention', marker='o', markersize=3)
plt.plot(losses_attn, label='Seq2Seq w/ Attention', marker='s', markersize=3)
plt.xlabel('Epoch (per 50)')
plt.ylabel('Loss')
plt.title('Training Loss: Simple RNN vs Attention')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Compare on same sentences
test_sentences = [
    'il n est pas aussi grand que son pere',
    'je suis trop fatigue pour conduire',
    'je suis desole si c est une question idiote',
    'elle est en train de peindre un tableau',
    'nous sommes tous ensemble a l instant',
    'vous etes trop maigre',
    'tu es matinal',
    'je suis reellement fiere de vous',
]

print("Comparison of translations:")
print("=" * 70)
for sent in test_sentences:
    words_rnn, _ = evaluate(encoder_rnn, decoder_rnn, sent, input_lang, output_lang)
    words_attn, _ = evaluate(encoder_attn, decoder_attn, sent, input_lang, output_lang)
    print(f"Input:   {sent}")
    print(f"No Attn: {' '.join(words_rnn)}")
    print(f"With Attn:{' '.join(words_attn)}")
    print("-" * 70)

## Visualizing Attention

A useful property of the attention mechanism is its interpretability — we can see where the network focuses at each decoding step.

In [ ]:
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap='bone')
    fig.colorbar(cax)

    input_words = input_sentence.split(' ')
    ax.set_xticks(range(attentions.shape[1]))
    ax.set_yticks(range(len(output_words)))
    ax.set_xticklabels(input_words + ['<EOS>'], rotation=90)
    ax.set_yticklabels(output_words)

    plt.xlabel('Input Words')
    plt.ylabel('Output Words')
    plt.title('Attention Visualization')
    plt.show()


def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(encoder_attn, decoder_attn,
                                         input_sentence, input_lang, output_lang)
    print('Input  =', input_sentence)
    print('Output =', ' '.join(output_words))
    showAttention(input_sentence, output_words,
                  attentions[0, :len(output_words), :])


evaluateAndShowAttention('il n est pas aussi grand que son pere')
evaluateAndShowAttention('je suis trop fatigue pour conduire')
evaluateAndShowAttention('je suis reellement fiere de vous')

## Analysis and Summary

### 1. Model Architecture
- **Encoder**: GRU-based RNN that processes the French input sentence, producing hidden states at each position.
- **Without Attention**: The decoder only receives the encoder's *final* hidden state (context vector), which must compress the entire input sentence meaning into a single fixed-size vector.
- **With Attention (Bahdanau)**: At each decoding step, the decoder computes attention weights over ALL encoder outputs, creating a dynamic context vector that focuses on relevant parts of the input.

### 2. Impact of Attention
- **Better translation quality**: Attention allows the model to align source words with target words, producing more accurate translations.
- **Handling longer sentences**: The simple RNN decoder struggles with longer inputs because the fixed context vector becomes a bottleneck; attention bypasses this by giving the decoder direct access to all encoder states.
- **Interpretability**: Attention weights show which input words the model focuses on when generating each output word, making the model's decisions more transparent.

### 3. Key Observations
- The attention model typically converges to a lower loss faster than the simple RNN model.
- The simple RNN decoder tends to produce more generic/repetitive outputs (e.g., "I'm sorry..."), while the attention model captures more specific content from the input.